# Type-discovery prompt + LLM test

This notebook tests the type-discovery prompt used by the type indexer:

1. Fill and inspect `TYPE_DISCOVERY_PROMPT`.
2. Send that prompt directly to a language model using `TypeProposals` as the structured response schema.

It does **not** import or instantiate the type indexer, and it does not access the database.

In [1]:
from pathlib import Path
import json
import os
import sys

from dotenv import load_dotenv

# Find the repository root even if Jupyter starts in a child directory.
working_dir = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (working_dir, *working_dir.parents)
     if (candidate / 'dotvectordb' / 'prompts' / 'type_discovery_prompts.py').is_file()),
    None,
)
assert repo_root is not None, 'Could not find the DotVectorDB repository root.'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / '.env')

from dotvectordb.prompts.type_discovery_prompts import (
    OUTPUT_EXAMPLE,
    TYPE_DISCOVERY_PROMPT,
    TypeProposals,
)

print(f'Repository: {repo_root}')

Repository: /home/x/DotVectorDB


## Choose the language-model provider

Set `PROVIDER` to `"openai"` or `"openrouter"`. Provider classes use the corresponding variables from `.env`, including the API key, model, tokenizer, and maximum-token settings. No secret values are printed.

In [2]:
PROVIDER = 'openai'  # Change to 'openrouter' if desired.

if PROVIDER == 'openai':
    from dotvectordb.plugins.llm_providers.openai_llms import OpenAiAgent
    required_key = 'OPENAI_API_KEY'
    language_model = OpenAiAgent()
elif PROVIDER == 'openrouter':
    from dotvectordb.plugins.llm_providers.openrouter_llms import OpenRouterAgent
    required_key = 'OPENROUTER_API_KEY'
    language_model = OpenRouterAgent()
else:
    raise ValueError("PROVIDER must be 'openai' or 'openrouter'.")

assert os.getenv(required_key), f'{required_key} is not set in the environment or .env file.'
print(f'Provider: {PROVIDER}; model: {language_model.model_name}')

Provider: openai; model: gpt-4o-mini


## Edit the test input

Replace `chunk` with any text whose entities, relationships, and proposed entity types you want to inspect.

In [3]:
chunk = """
Ada Lovelace collaborated with Charles Babbage on the Analytical Engine in London.
Her notes describing how the machine could calculate Bernoulli numbers are often
regarded as an early computer program.
""".strip()

input_example = (
    'Marie Curie conducted research on radioactivity at the University of Paris.'
)

print(chunk)

Ada Lovelace collaborated with Charles Babbage on the Analytical Engine in London.
Her notes describing how the machine could calculate Bernoulli numbers are often
regarded as an early computer program.


## Build and inspect the exact prompt

The prompt template currently contains three placeholders, including `input_example`. This cell fills all three so the prompt can be tested independently.

In [4]:
filled_prompt = TYPE_DISCOVERY_PROMPT.format(
    chunk=chunk,
    input_example=input_example,
    output_example=json.dumps(OUTPUT_EXAMPLE, indent=2),
)

assert '{chunk}' not in filled_prompt
assert '{input_example}' not in filled_prompt
assert '{output_example}' not in filled_prompt
print(filled_prompt)


<Role>
You are a Knowledge Graph Specialist responsible for reading the chunk from the '<Input>' section and extracting as many relationships between the entities in the text as possible. You will also invent appropriate entity types for each entity based on the context of the input text. Try to stay consistent if entity types are reusable. 
</Role>

<Steps>
1. Read the '<Input>' section carefully 
2. Read the '<DEFAULT_ENTITY_TYPES>' section to understand the default entity types that you can use as a reference for inventing new entity types. **Only invent a new entity type if it is not covered by the default entity types (e.g, DO NOT, I REPEAT, DO NOT CREATE CITY OR COUNTRY types if LOCATION is already a default type)**
3. Then, identify all unique relationships that seem important to understanding the input text. For each relationship, extract the following information:
- src_entity: The name of the source entity in the relationship. Ensure **consistent naming** across the entire e

## Call the language model

Running the next cell sends the displayed prompt directly to the selected provider. It makes a real API request and may incur provider charges.

In [5]:
response, total_tokens = await language_model.generate(
    filled_prompt,
    structuredInput=TypeProposals,
)

X = response["Proposals"]

for proposal in X:
    print(f"Source: {proposal['src_entity']}")
    print(f"Destination: {proposal['dst_entity']}")
    print(f"Source Types: {proposal['src_entity_types']}")
    print(f"Destination Types: {proposal['dst_entity_types']}")



print('Structured response:')
print(json.dumps(response, indent=2))
print(f'\nTotal tokens: {total_tokens}')

Source: Ada Lovelace
Destination: Charles Babbage
Source Types: ['person']
Destination Types: ['person']
Source: Ada Lovelace
Destination: Analytical Engine
Source Types: ['person']
Destination Types: ['technology']
Source: Analytical Engine
Destination: London
Source Types: ['technology']
Destination Types: ['location']
Source: Ada Lovelace
Destination: Bernoulli numbers
Source Types: ['person']
Destination Types: ['mathematical_concept']
Source: Ada Lovelace's notes
Destination: early computer program
Source Types: ['work_of_art']
Destination Types: ['concept']
Structured response:
{
  "Proposals": [
    {
      "src_entity": "Ada Lovelace",
      "src_entity_types": [
        "person"
      ],
      "dst_entity": "Charles Babbage",
      "dst_entity_types": [
        "person"
      ]
    },
    {
      "src_entity": "Ada Lovelace",
      "src_entity_types": [
        "person"
      ],
      "dst_entity": "Analytical Engine",
      "dst_entity_types": [
        "technology"
      ]
 

## Optional schema inspection

Use this to see the exact structured-output schema sent to the provider. It is useful when the API rejects the schema or the response keys differ from the prompt examples.

In [6]:
schema = (
    TypeProposals.model_json_schema()
    if hasattr(TypeProposals, 'model_json_schema')
    else TypeProposals.schema()
)
print(json.dumps(schema, indent=2))

{
  "$defs": {
    "Relationships": {
      "properties": {
        "src_entity": {
          "title": "Src Entity",
          "type": "string"
        },
        "entity_types": {
          "items": {
            "type": "string"
          },
          "title": "Entity Types",
          "type": "array"
        },
        "dst_entity": {
          "title": "Dst Entity",
          "type": "string"
        }
      },
      "required": [
        "src_entity",
        "entity_types",
        "dst_entity"
      ],
      "title": "Relationships",
      "type": "object"
    }
  },
  "properties": {
    "Proposals": {
      "items": {
        "$ref": "#/$defs/Relationships"
      },
      "title": "Proposals",
      "type": "array"
    }
  },
  "required": [
    "Proposals"
  ],
  "title": "TypeProposals",
  "type": "object"
}
